### Imports

In [ ]:
import cv2
import h5py
import torch
import numpy as np
from torch import nn, Tensor
from tqdm import tqdm
import torch.nn.functional as F
from segment_anything.modeling.sam import Sam
from segment_anything import sam_model_registry
from torch.utils.data import Dataset, DataLoader
from scipy.spatial.distance import directed_hausdorff

## Training

### Data Handling

In [ ]:
def load_lits_data(train_path, val_path, test_path):
    """Load LiTS dataset from HDF5 files."""
    with h5py.File(train_path, "r") as file:
        x_train = np.array([file[f"Slice{str(i)}"]["Slice"][:] for i in range(len(file))])
        y_train = np.array([file[f"Slice{str(i)}"]["Segmentation"][:] for i in range(len(file))])

    with h5py.File(val_path, "r") as file:
        x_val = np.array([file[f"Slice{str(i)}"]["Slice"][:] for i in range(len(file))])
        y_val = np.array([file[f"Slice{str(i)}"]["Segmentation"][:] for i in range(len(file))])

    with h5py.File(test_path, "r") as file:
        x_test = np.array([file[f"Slice{str(i)}"]["Slice"][:] for i in range(len(file))])
        y_test = np.array([file[f"Slice{str(i)}"]["Segmentation"][:] for i in range(len(file))])

    # Process masks
    y_train = np.expand_dims(y_train, axis = 1)
    y_val = np.expand_dims(y_val, axis = 1)
    y_test = np.expand_dims(y_test, axis = 1)

    # Convert class 2 (tumor) to class 1 (liver)
    y_train = np.where(y_train == 2, 1, y_train)
    y_val = np.where(y_val == 2, 1, y_val)
    y_test = np.where(y_test == 2, 1, y_test)

    return (x_train, y_train), (x_val, y_val), (x_test, y_test)

def create_classification_labels(segmentation_masks):
    """Create binary classification labels from segmentation masks."""
    return np.array([1 if np.any(mask > 0) else 0 for mask in segmentation_masks])

class MultitaskLiTSDataset(Dataset):
    """Dataset class for multitask learning with classification and segmentation."""
    def __init__(self, images, masks, phase = "train", num_seg_samples = 50, image_size = 1024):
        self.images = images
        self.masks = masks
        self.phase = phase
        self.image_size = image_size

        print(f"\nInitializing {phase} dataset...")

        # Create classification labels
        self.classification_labels = create_classification_labels(masks)

        # Select samples for segmentation training (positives only)
        positive_indices = np.where(self.classification_labels == 1)[0]

        if phase == "train":
            self.seg_indices = set(
                np.random.choice(
                    positive_indices,
                    min(num_seg_samples, len(positive_indices)),
                    replace = False,
                )
            )
        elif phase in {"val", "test"}:
            # Use ALL positive masks for segmentation during validation and test
            self.seg_indices = set(positive_indices.tolist())
        else:
            self.seg_indices = set()

        print(f"{phase} dataset created with {len(self.images)} samples, {len(self.seg_indices)} for segmentation")
        print()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Get original image and ensure it's 2D
        image_orig = self.images[idx]
        if isinstance(image_orig, np.ndarray):
            image_orig = image_orig.squeeze()
        else:
            image_orig = np.array(image_orig).squeeze()

        # Prepare SAM image (normalized to [0, 1])
        image_sam = image_orig.astype(np.float32)
        image_sam = (image_sam - image_sam.min()) / (image_sam.max() - image_sam.min() + 1e-8)

        # Resize using linear interpolation
        image_sam = cv2.resize(
            image_sam.astype(np.float32),
            (self.image_size, self.image_size),
            interpolation = cv2.INTER_LINEAR,
        )

        # Convert to RGB by stacking
        image_sam = np.stack([image_sam] * 3, axis = -1)

        # Get mask and convert to binary
        mask = self.masks[idx].squeeze().astype(np.float32)

        # Resize mask
        mask_sam = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation = cv2.INTER_NEAREST,
        )
        mask_sam = (mask_sam > 0.5).astype(np.float32)

        # Get classification label
        label = self.classification_labels[idx]

        # Flag for segmentation supervision (GT loss)
        use_for_seg = idx in self.seg_indices

        return {
            "image_sam": torch.FloatTensor(image_sam.transpose(2, 0, 1)),
            "label": torch.LongTensor([label])[0],
            "mask": torch.FloatTensor(mask_sam),  # always returned (but only used for loss if use_for_seg)
            "use_for_seg": torch.tensor(use_for_seg),  # collates to torch.bool
        }

def create_multitask_dataloaders(batch_size = 30):
    """Create DataLoaders for training, validation, and testing."""
    (x_train, y_train), (x_val, y_val), (x_test, y_test) = load_lits_data(
        "/localdisk1/Datasets/LiTS/Processed/Binary/FullLiTSTrainingDataset.hdf5",
        "/localdisk1/Datasets/LiTS/Processed/Binary/FullLiTSValidationDataset.hdf5",
        "/localdisk1/Datasets/LiTS/Processed/Binary/FullLiTSTestingDataset.hdf5",
    )

    train_dataset = MultitaskLiTSDataset(x_train, y_train, phase = "train", num_seg_samples = 50)
    val_dataset = MultitaskLiTSDataset(x_val, y_val, phase = "val")
    test_dataset = MultitaskLiTSDataset(x_test, y_test, phase = "test")

    dataloaders = {
        "train": DataLoader(train_dataset, batch_size = batch_size, shuffle = True),
        "val": DataLoader(val_dataset, batch_size = batch_size),
        "test": DataLoader(test_dataset, batch_size = batch_size),
    }

    return dataloaders

### Loss Functions

In [ ]:
class DiceLoss(nn.Module):
    """Dice loss for segmentation tasks."""
    def __init__(self, smooth = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, predicted, target):
        # predicted: [N, 1, H, W], target: [N, 1, H, W]
        batch_size = predicted.size(0)
        predicted_flat = predicted.view(batch_size, -1)
        target_flat = target.view(batch_size, -1)

        intersection = (predicted_flat * target_flat).sum(1)
        union = predicted_flat.sum(1) + target_flat.sum(1)

        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

class MultitaskLoss(nn.Module):
    """Combined loss for classification and segmentation (seg loss only on supervised subset)."""
    def __init__(self, seg_weight = 0.5):
        super().__init__()
        self.classification_loss = nn.CrossEntropyLoss()
        self.segmentation_loss = DiceLoss()
        self.seg_weight = seg_weight

    def forward(self, cls_pred, cls_target, seg_pred = None, seg_target = None):
        cls_loss = self.classification_loss(cls_pred, cls_target)

        if seg_pred is not None and seg_target is not None:
            seg_loss = self.segmentation_loss(seg_pred, seg_target)
            return cls_loss + self.seg_weight * seg_loss

        return cls_loss

### LoRA Components

In [ ]:
class LoRA_qkv(nn.Module):
    """LoRA module for query-key-value attention layers."""

    def __init__(self, qkv, linear_a_q, linear_b_q, linear_a_v, linear_b_v):
        super().__init__()
        self.qkv = qkv
        self.linear_a_q = linear_a_q
        self.linear_b_q = linear_b_q
        self.linear_a_v = linear_a_v
        self.linear_b_v = linear_b_v
        self.d_model = qkv.in_features

    def forward(self, x: Tensor):
        qkv = self.qkv(x)
        q_ba = self.linear_b_q(self.linear_a_q(x))
        v_ba = self.linear_b_v(self.linear_a_v(x))

        # Add LoRA adaptations to q and v
        qkv[:, :, :, : self.d_model] += q_ba
        qkv[:, :, :, -self.d_model :] += v_ba

        return qkv

class LoRA_sam(nn.Module):
    """LoRA wrapper for SAM model."""

    def __init__(self, sam_model: Sam, rank: int, lora_layer = None):
        super().__init__()

        self.rank = rank
        if self.rank <= 0:
            raise ValueError("rank must be > 0")

        if lora_layer:
            self.lora_layer = lora_layer
        else:
            self.lora_layer = list(range(len(sam_model.image_encoder.blocks)))

        self.A_weights = []
        self.B_weights = []

        # Freeze parameters of the image encoder
        for param in sam_model.image_encoder.parameters():
            param.requires_grad = False

        # Add LoRA layers
        for t_layer_i, blk in enumerate(sam_model.image_encoder.blocks):
            if t_layer_i not in self.lora_layer:
                continue

            w_qkv_linear = blk.attn.qkv
            self.d_model = w_qkv_linear.in_features

            w_a_linear_q = nn.Linear(self.d_model, self.rank, bias = False)
            w_b_linear_q = nn.Linear(self.rank, self.d_model, bias = False)
            w_a_linear_v = nn.Linear(self.d_model, self.rank, bias = False)
            w_b_linear_v = nn.Linear(self.rank, self.d_model, bias = False)

            self.A_weights.extend([w_a_linear_q, w_a_linear_v])
            self.B_weights.extend([w_b_linear_q, w_b_linear_v])

            blk.attn.qkv = LoRA_qkv(
                w_qkv_linear,
                w_a_linear_q,
                w_b_linear_q,
                w_a_linear_v,
                w_b_linear_v,
            )

        # Register parameters
        for i, (w_a, w_b) in enumerate(zip(self.A_weights, self.B_weights)):
            self.register_parameter(f"w_a_{i}", w_a.weight)
            self.register_parameter(f"w_b_{i}", w_b.weight)

        self.reset_parameters()
        self.sam = sam_model
        self.lora_vit = sam_model.image_encoder

    def reset_parameters(self):
        for w_A in self.A_weights:
            nn.init.normal_(w_A.weight, mean = 0.0, std = 0.01)
        for w_B in self.B_weights:
            nn.init.zeros_(w_B.weight)

### Prompting necessities

In [ ]:
def generate_boxes(masks_2d: torch.Tensor) -> torch.Tensor:
    """
    Generate bounding boxes from masks.
    """
    boxes = []
    for mask in masks_2d:
        yx = (mask > 0.5).nonzero(as_tuple = False)  # [K, 2] where columns are (y, x)
        if yx.numel() > 0:
            y_min = yx[:, 0].min().item()
            y_max = yx[:, 0].max().item()
            x_min = yx[:, 1].min().item()
            x_max = yx[:, 1].max().item()
            boxes.append(torch.tensor([x_min, y_min, x_max, y_max], dtype=torch.float32, device=masks_2d.device))
        else:
            h, w = mask.shape
            boxes.append(torch.tensor([w / 4.0, h / 4.0, 3.0 * w / 4.0, 3.0 * h / 4.0], dtype=torch.float32, device=masks_2d.device))

    return torch.stack(boxes, dim = 0)

def cams_to_boxes(cams: torch.Tensor, threshold: float = 0.4) -> torch.Tensor:
    """
    Convert GradCAM heatmaps to bounding boxes.
    """
    boxes = []
    for cam in cams:
        yx = (cam >= threshold).nonzero(as_tuple = False)  # [K, 2] (y, x)
        if yx.numel() > 0:
            y_min = yx[:, 0].min().item()
            y_max = yx[:, 0].max().item()
            x_min = yx[:, 1].min().item()
            x_max = yx[:, 1].max().item()
            boxes.append(torch.tensor([x_min, y_min, x_max, y_max], dtype=torch.float32, device=cams.device))
        else:
            h, w = cam.shape
            boxes.append(torch.tensor([w / 4.0, h / 4.0, 3.0 * w / 4.0, 3.0 * h / 4.0], dtype=torch.float32, device=cams.device))

    return torch.stack(boxes, dim = 0)

def gradcam_from_embedding(image_embedding: torch.Tensor, class_scores: torch.Tensor) -> torch.Tensor:
    """
    Compute GradCAM on the embedding tensor.
    """
    grads = torch.autograd.grad(
        outputs = class_scores.sum(),
        inputs = image_embedding,
        retain_graph = True,
        create_graph = False,
        allow_unused = False,
    )[0]  # [B, C, H, W]

    weights = grads.mean(dim = (2, 3), keepdim = True)  # [B, C, 1, 1]
    cam = (weights * image_embedding).sum(dim = 1)  # [B, H, W]
    cam = F.relu(cam)

    # Normalize per-sample to [0, 1]
    cam_min = cam.amin(dim = (1, 2), keepdim = True)
    cam_max = cam.amax(dim = (1, 2), keepdim = True)
    cam = (cam - cam_min) / (cam_max - cam_min + 1e-6)

    return cam.detach()

### ADA-SAM

In [ ]:
class MultitaskSAM(nn.Module):
    """
    - Training with use_for_seg=True: boxes come from GT masks.
    - Training with use_for_seg=False: boxes come from GradCAM.
    """

    def __init__(self, sam_checkpoint, model_type = "vit_b", num_classes = 2, cam_threshold = 0.4):
        super().__init__()

        # Initialize SAM model with LoRA
        self.sam = sam_model_registry[model_type](checkpoint = sam_checkpoint)
        sam_lora_class = LoRA_sam(self.sam, rank = 8)
        self.sam_lora = sam_lora_class.sam

        sam_embed_dim = 256

        # Classification head
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(sam_embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

        self.cam_threshold = cam_threshold

    def forward(self, batch):
        images_sam = batch["image_sam"]  # [B, 3, 1024, 1024]
        use_seg = batch["use_for_seg"]  # [B] bool tensor
        masks = batch["mask"]  # [B, 1024, 1024] (always present, but only supervised on use_seg)

        if use_seg.dtype != torch.bool:
            use_seg = use_seg.bool()

        # Shared image embeddings
        image_embedding = self.sam_lora.image_encoder(images_sam)  # [B, 256, H, W]

        # Classification branch
        pooled_features = self.global_pool(image_embedding).squeeze(-1).squeeze(-1)  # [B, 256]
        cls_output = self.classifier(pooled_features)  # [B, 2]

        # Determine box strategy based on training mode
        if self.training:
            # TRAINING MODE:
            # - use_for_seg=True: use GT boxes
            # - use_for_seg=False: use GradCAM boxes
            
            b, _, h_e, w_e = image_embedding.shape
            scale_x = images_sam.shape[3] / float(w_e)
            scale_y = images_sam.shape[2] / float(h_e)

            # Initialize with GradCAM boxes for all samples
            target_scores = cls_output[:, 1]  # [B]
            cams = gradcam_from_embedding(image_embedding, target_scores)  # [B, H_e, W_e]
            cam_boxes = cams_to_boxes(cams, threshold = self.cam_threshold)  # [B, 4] on embedding resolution
            
            # Scale to input resolution
            cam_boxes_scaled = cam_boxes.clone()
            cam_boxes_scaled[:, 0] = cam_boxes[:, 0] * scale_x
            cam_boxes_scaled[:, 2] = cam_boxes[:, 2] * scale_x
            cam_boxes_scaled[:, 1] = cam_boxes[:, 1] * scale_y
            cam_boxes_scaled[:, 3] = cam_boxes[:, 3] * scale_y

            boxes_final = cam_boxes_scaled

            # Replace with GT boxes for supervised samples
            if use_seg.any():
                sup_idx = use_seg.nonzero(as_tuple = False).squeeze(1)  # [N_sup]
                gt_boxes = generate_boxes(masks[sup_idx])  # [N_sup, 4] already in 1024 coords
                boxes_final = boxes_final.clone()
                boxes_final[sup_idx] = gt_boxes
            else:
                sup_idx = None

        else:
            boxes_final = generate_boxes(masks)
            sup_idx = use_seg.nonzero(as_tuple = False).squeeze(1) if use_seg.any() else None

        # Segmentation branch runs for ALL samples, prompted by boxes_final
        sparse_embeddings, dense_embeddings = self.sam_lora.prompt_encoder(
            points = None,
            boxes = boxes_final,
            masks = None,
        )

        mask_predictions, _ = self.sam_lora.mask_decoder(
            image_embeddings = image_embedding,
            image_pe = self.sam_lora.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings = sparse_embeddings,
            dense_prompt_embeddings = dense_embeddings,
            multimask_output = False,
        )  # [B, 1, 256, 256]

        upscaled_masks = F.interpolate(
            mask_predictions,
            size = (images_sam.shape[2], images_sam.shape[3]),
            mode = "bilinear",
            align_corners = False,
        )

        seg_output = torch.sigmoid(upscaled_masks)  # [B, 1, 1024, 1024]

        return cls_output, seg_output, sup_idx

### Training loop

In [ ]:
def train_multitask_model(model, train_loader, val_loader, num_epochs = 100, device = "cuda"):
    model = model.to(device)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr = 1e-4,
    )

    criterion = MultitaskLoss(seg_weight = 0.5)
    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        train_pbar = tqdm(
            enumerate(train_loader),
            total = len(train_loader),
            desc = f"Epoch {epoch + 1}/{num_epochs} [Train]",
            leave = True,
        )

        for batch_idx, batch in train_pbar:
            optimizer.zero_grad()

            batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

            cls_output, seg_output, sup_idx = model(batch)

            # Classification loss always
            if sup_idx is not None and sup_idx.numel() > 0:
                # Seg loss only on supervised subset
                seg_pred_sup = seg_output[sup_idx]  # [N, 1, H, W]
                seg_tgt_sup = batch["mask"][sup_idx].unsqueeze(1).float()  # [N, 1, H, W]
                loss = criterion(cls_output, batch["label"], seg_pred_sup, seg_tgt_sup)
            else:
                loss = criterion(cls_output, batch["label"])

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_pbar.set_postfix({"loss": f"{loss.item():.4f}", "avg_loss": f"{train_loss / (batch_idx + 1):.4f}"})

        model.eval()
        val_loss = 0.0

        val_pbar = tqdm(
            enumerate(val_loader),
            total = len(val_loader),
            desc = f"Epoch {epoch + 1}/{num_epochs} [Val]",
            leave = True,
        )

        with torch.no_grad():  # Changed from torch.enable_grad() to torch.no_grad()
            for batch_idx, batch in val_pbar:
                batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

                cls_output, seg_output, sup_idx = model(batch)

                if sup_idx is not None and sup_idx.numel() > 0:
                    seg_pred_sup = seg_output[sup_idx]
                    seg_tgt_sup = batch["mask"][sup_idx].unsqueeze(1).float()
                    loss = criterion(cls_output, batch["label"], seg_pred_sup, seg_tgt_sup)
                else:
                    loss = criterion(cls_output, batch["label"])

                val_loss += loss.item()
                val_pbar.set_postfix({"loss": f"{loss.item():.4f}", "avg_loss": f"{val_loss / (batch_idx + 1):.4f}"})

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                },
                "best_multitask_model_cam_prompted.pth",
            )
            print(f"✓ New best model saved! Val Loss: {best_val_loss:.4f}")

        print(f"\nEpoch {epoch + 1}/{num_epochs} Summary:")
        print(f"  Training Loss: {avg_train_loss:.4f}")
        print(f"  Validation Loss: {avg_val_loss:.4f}")
        print("-" * 50)

### Main function - training

In [ ]:
def main():
    device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    dataloaders = create_multitask_dataloaders(batch_size = 1)

    sam_checkpoint = "/localdisk0/Tyler_Deprecated/SAM-Mix/sam_vit_b_01ec64.pth"
    model = MultitaskSAM(sam_checkpoint, cam_threshold = 0.4)

    print("\n" + "=" * 50)
    print("Model Architecture:")
    print("=" * 50)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters: {total_params - trainable_params:,}")
    print("=" * 50 + "\n")

    train_multitask_model(
        model = model,
        train_loader = dataloaders["train"],
        val_loader = dataloaders["val"],
        num_epochs = 10,
        device = device,
    )
        val_loader=dataloaders['val'],
        num_epochs=10,
        device=device
    )

if __name__ == "__main__":
    main()

## Evaluation

### Test data handling

In [ ]:
def load_lits_test_data(test_path):
    with h5py.File(test_path, "r") as file:
        x_test = np.array([
            file[f"Slice{str(i)}"]["Slice"][:]
            for i in range(len(file))
        ])
        y_test = np.array([
            file[f"Slice{str(i)}"]["Segmentation"][:]
            for i in range(len(file))
        ])

    y_test = np.expand_dims(y_test, axis = 1)
    y_test = np.where(y_test == 2, 1, y_test)

    return x_test, y_test


def create_classification_labels(segmentation_masks):
    return np.array([
        1 if np.any(mask > 0) else 0
        for mask in segmentation_masks
    ])

class EvalLiTSSegmentationDataset(Dataset):
    def __init__(self, images, masks, image_size = 1024):
        self.images = images
        self.masks = masks
        self.image_size = image_size

        self.class_labels = create_classification_labels(masks)
        self.indices = np.where(self.class_labels == 1)[0]

        print(
            f"Evaluation dataset initialized with "
            f"{len(self.indices)} segmentation samples"
        )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]

        image = self.images[real_idx].squeeze().astype(np.float32)
        image = (image - image.min()) / (image.max() - image.min() + 1e-8)

        image = cv2.resize(
            image,
            (self.image_size, self.image_size),
            interpolation = cv2.INTER_LINEAR
        )

        image = np.stack([image] * 3, axis = -1)

        mask = self.masks[real_idx].squeeze().astype(np.float32)

        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation = cv2.INTER_NEAREST
        )

        mask = (mask > 0.5).astype(np.float32)

        return {
            "image_sam": torch.from_numpy(image).permute(2, 0, 1),
            "mask": torch.from_numpy(mask),
            "use_for_seg": torch.tensor(True),
            "label": torch.tensor(1, dtype = torch.long)
        }

def create_eval_dataloader(batch_size = 1):
    x_test, y_test = load_lits_test_data(
        "/localdisk1/Datasets/LiTS/Processed/Binary/FullLiTSTestingDataset.hdf5"
    )

    dataset = EvalLiTSSegmentationDataset(x_test, y_test)

    return DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = False
    )

### Metrics

In [ ]:
def dice_score(pred, target, eps = 1e-6):
    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()

    return ((2.0 * intersection + eps) / (union + eps)).item()


def hausdorff_distance(pred, target):
    pred_pts = np.argwhere(pred > 0)
    target_pts = np.argwhere(target > 0)

    if len(pred_pts) == 0 and len(target_pts) == 0:
        return 0.0
    if len(pred_pts) == 0 or len(target_pts) == 0:
        return np.inf

    d1 = directed_hausdorff(pred_pts, target_pts)[0]
    d2 = directed_hausdorff(target_pts, pred_pts)[0]

    return max(d1, d2)

### Evaluation loop

In [ ]:
def evaluate_segmentation(model, dataloader, device, threshold = 0.5):
    model.eval()
    model.to(device)

    dice_scores = []
    hausdorff_scores = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc = "Evaluating segmentation"):
            batch = {
                k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()
            }

            _, seg_out, seg_indices = model(batch)

            if seg_out is None:
                continue

            gt_masks = batch["mask"][seg_indices]

            for i in range(seg_out.shape[0]):
                pred_mask = (seg_out[i, 0] > threshold).float()
                gt_mask = gt_masks[i]

                dice_scores.append(dice_score(pred_mask, gt_mask))
                hausdorff_scores.append(
                    hausdorff_distance(
                        pred_mask.cpu().numpy(),
                        gt_mask.cpu().numpy()
                    )
                )

    return {
        "num_samples": len(dice_scores),
        "mean_dice": float(np.mean(dice_scores)) if dice_scores else 0.0,
        "std_dice": float(np.std(dice_scores)) if dice_scores else 0.0,
        "mean_hausdorff": float(np.mean(hausdorff_scores)) if hausdorff_scores else np.inf,
        "std_hausdorff": float(np.std(hausdorff_scores)) if hausdorff_scores else np.inf,
    }

### Main function - evaluation

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    eval_loader = create_eval_dataloader(batch_size = 1)

    sam_checkpoint = "/localdisk0/Tyler_Deprecated/SAM-Mix/sam_vit_b_01ec64.pth"
    model = MultitaskSAM(sam_checkpoint)

    checkpoint = torch.load(
        "best_multitask_model_integrated_50_using_seg_for_val.pth",
        map_location = device
    )
    model.load_state_dict(checkpoint["model_state_dict"])

    results = evaluate_segmentation(
        model = model,
        dataloader = eval_loader,
        device = device
    )

    print("\nSegmentation Evaluation Results")
    print(f"Evaluated samples: {results['num_samples']}")
    print(f"Dice: {results['mean_dice']:.4f} ± {results['std_dice']:.4f}")
    print(f"Hausdorff: {results['mean_hausdorff']:.2f} ± {results['std_hausdorff']:.2f}")


if __name__ == "__main__":
    main()